# Market Exploration

Explore Kalshi market data stored in DuckDB. Analyze categories, volume distributions,
spread patterns, and identify opportunities.

In [ ]:
import sys
sys.path.insert(0, '../src')

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timezone, timedelta

DATA_DIR = Path('../data')
MARKET_DB = DATA_DIR / 'market_data.duckdb'

con = duckdb.connect(str(MARKET_DB), read_only=True)
print('Tables:', con.execute("SHOW TABLES").fetchall())

## Market Overview

In [ ]:
# Count markets by category
df_categories = con.execute("""
    SELECT category, COUNT(DISTINCT ticker) as market_count,
           SUM(volume) as total_volume,
           AVG(yes_ask - yes_bid) as avg_spread
    FROM market_states
    WHERE recorded_at > now() - INTERVAL '24 hours'
    GROUP BY category
    ORDER BY total_volume DESC
""").df()
df_categories

In [ ]:
# Most active markets by volume
df_active = con.execute("""
    SELECT ticker, title, category,
           MAX(volume) as volume,
           MAX(open_interest) as open_interest,
           AVG(yes_ask - yes_bid) as avg_spread,
           AVG(last_price) as avg_price
    FROM market_states
    WHERE recorded_at > now() - INTERVAL '24 hours'
      AND status = 'open'
    GROUP BY ticker, title, category
    ORDER BY volume DESC
    LIMIT 20
""").df()
df_active

## Spread Analysis

In [ ]:
# Spread distribution across markets
df_spreads = con.execute("""
    SELECT ticker, category,
           AVG(yes_ask - yes_bid) as avg_spread,
           MIN(yes_ask - yes_bid) as min_spread,
           MAX(yes_ask - yes_bid) as max_spread,
           STDDEV(yes_ask - yes_bid) as spread_volatility,
           COUNT(*) as observations
    FROM market_states
    WHERE recorded_at > now() - INTERVAL '7 days'
      AND yes_bid > 0 AND yes_ask > 0
    GROUP BY ticker, category
    HAVING COUNT(*) > 10
    ORDER BY avg_spread ASC
""").df()
print(f'Markets with data: {len(df_spreads)}')
print(f'\nTightest spreads:')
df_spreads.head(10)

## Orderbook Depth Analysis

In [ ]:
# Orderbook snapshots - depth over time
df_ob = con.execute("""
    SELECT ticker,
           AVG(json_array_length(yes_levels::JSON)) as avg_yes_levels,
           AVG(json_array_length(no_levels::JSON)) as avg_no_levels,
           COUNT(*) as snapshots
    FROM orderbook_snapshots
    WHERE recorded_at > now() - INTERVAL '24 hours'
    GROUP BY ticker
    HAVING COUNT(*) > 5
    ORDER BY avg_yes_levels DESC
    LIMIT 20
""").df()
df_ob

## Price Movement Patterns

In [ ]:
# Price time series for a specific market
# Change the ticker to explore different markets
TICKER = df_active['ticker'].iloc[0] if len(df_active) > 0 else 'EXAMPLE'

df_prices = con.execute(f"""
    SELECT recorded_at, last_price, yes_bid, yes_ask, volume
    FROM market_states
    WHERE ticker = '{TICKER}'
    ORDER BY recorded_at
""").df()

if len(df_prices) > 0:
    print(f'Ticker: {TICKER}')
    print(f'Observations: {len(df_prices)}')
    print(f'Price range: {df_prices["last_price"].min():.4f} - {df_prices["last_price"].max():.4f}')
    print(f'Current spread: {(df_prices["yes_ask"].iloc[-1] - df_prices["yes_bid"].iloc[-1]):.4f}')
else:
    print('No data yet - run the market_data worker first')

## Sportsbook Line Comparison

In [ ]:
# Check sportsbook data from collector DB
COLLECTOR_DB = DATA_DIR / 'collector.duckdb'
if COLLECTOR_DB.exists():
    con2 = duckdb.connect(str(COLLECTOR_DB), read_only=True)
    df_lines = con2.execute("""
        SELECT league, COUNT(*) as records,
               COUNT(DISTINCT game_id) as games,
               MIN(captured_at) as first_capture,
               MAX(captured_at) as last_capture
        FROM sportsbook_game_lines
        GROUP BY league
    """).df()
    print('Sportsbook data collected:')
    display(df_lines)
    con2.close()
else:
    print('No collector database yet')

In [ ]:
con.close()
print('Done.')